In [1]:
import pandas as pd
import re
import os
import hashlib
from datetime import datetime
from typing import Dict
import nltk
from nltk.corpus import stopwords

In [2]:
# Define the input path for the raw Enron dataset and the output directory for processed files.
INPUT_CSV = r"C:\Users\pawan\Desktop\AI_KBG\datasets\emails.csv"
OUTPUT_DIR = "final_datasets"
LOG_DIR = "logs"
EMAIL_PATTERN = re.compile(r"^[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}$")

print("--- Starting Full Interactive Pipeline ---")

--- Starting Full Interactive Pipeline ---


In [3]:
# --- NLP Setup ---
# Downloads common word lists to help with text analysis later.
try:
    nltk.data.find("corpora/stopwords")
except LookupError:
    print("Downloading stopwords...")
    nltk.download("stopwords", quiet=True)
STOPWORDS = set(stopwords.words("english"))

In [4]:
# --- Step 1: Data Loading ---
print("\nStep 1: Loading dataset...")
df = None
for encoding in ["utf-8", "latin-1", "iso-8859-1", "cp1252"]:
    try:
        df = pd.read_csv(INPUT_CSV, encoding=encoding, low_memory=False)
        print(f"Loaded {len(df):,} records using {encoding} encoding.")
        break
    except UnicodeDecodeError:
        continue

if df is None:
    raise ValueError("Could not load the dataset with any standard encoding.")

# Visual check of the raw data.
print("\nSample of loaded data:")
df.head()



Step 1: Loading dataset...
Loaded 517,401 records using utf-8 encoding.

Sample of loaded data:


,file,message
0,allen-p/_sent_mail/1.,Message-ID: <18782981.1075855378110.JavaMail.e...
1,allen-p/_sent_mail/10.,Message-ID: <15464986.1075855378456.JavaMail.e...
2,allen-p/_sent_mail/100.,Message-ID: <24216240.1075855687451.JavaMail.e...
3,allen-p/_sent_mail/1000.,Message-ID: <13505866.1075863688222.JavaMail.e...
4,allen-p/_sent_mail/1001.,Message-ID: <30922949.1075863688243.JavaMail.e...


In [5]:
# --- Step 2: Email Parsing ---
print("\nStep 2: Parsing email headers and body...")
# Splits the raw 'message' field into structured headers and body text.
parsed = df["message"].str.split("\n\n", n=1, expand=True)
headers_block = parsed[0].fillna("")  # Top part contains headers like From, To, Date.
df["body_raw"] = parsed[1].fillna("")  # Bottom part contains the actual email text.

header_patterns = {
    "message_id": r"^Message-ID:\s*(.+?)$",
    "date": r"^Date:\s*(.+?)$",
    "from": r"^From:\s*(.+?)$",
    "to": r"^To:\s*(.+?)$",
    "subject": r"^Subject:\s*(.+?)$",
    "x_cc": r"^X-cc:\s*(.+?)$",
    "x_bcc": r"^X-bcc:\s*(.+?)$",
    "x_from": r"^X-From:\s*(.+?)$",
    "x_to": r"^X-To:\s*(.+?)$",
}

for col_name, regex in header_patterns.items():
    df[col_name] = headers_block.str.extract(regex, flags=re.MULTILINE | re.IGNORECASE)
    df[col_name] = df[col_name].str.strip().fillna("")

# Match character for character: Strip whitespace, fill missing
df['subject'] = df['subject'].fillna("").str.strip()

# Handle strictly empty subjects or subjects that only contain "Re:" / "FW:" via filtering
invalid_patterns = ['^$', '^Re:$', '^FW:$', '^Re: $', '^No Subject$']
df = df[~df['subject'].str.match('|'.join(invalid_patterns), case=False, na=False)]
df = df[df['subject'].str.len() >= 2]

print("Sample parsed content:")
df[["message_id","date","from", "to", "subject","body_raw"]].head()


Step 2: Parsing email headers and body...
Sample parsed content:


,message_id,date,from,to,subject,body_raw
0,<18782981.1075855378110.JavaMail.evans@thyme>,"Mon, 14 May 2001 16:39:00 -0700 (PDT)",phillip.allen@enron.com,tim.belden@enron.com,Mime-Version: 1.0,Here is our forecast\n\n
2,<24216240.1075855687451.JavaMail.evans@thyme>,"Wed, 18 Oct 2000 03:00:00 -0700 (PDT)",phillip.allen@enron.com,leah.arsdall@enron.com,Re: test,test successful. way to go!!!
3,<13505866.1075863688222.JavaMail.evans@thyme>,"Mon, 23 Oct 2000 06:13:00 -0700 (PDT)",phillip.allen@enron.com,randall.gay@enron.com,Mime-Version: 1.0,"Randy,\n\n Can you send me a schedule of the s..."
4,<30922949.1075863688243.JavaMail.evans@thyme>,"Thu, 31 Aug 2000 05:07:00 -0700 (PDT)",phillip.allen@enron.com,greg.piper@enron.com,Re: Hello,Let's shoot for Tuesday at 11:45.
5,<30965995.1075863688265.JavaMail.evans@thyme>,"Thu, 31 Aug 2000 04:17:00 -0700 (PDT)",phillip.allen@enron.com,greg.piper@enron.com,Re: Hello,"Greg,\n\n How about either next Tuesday or Thu..."


In [6]:
# --- Step 3: Data Cleaning ---
print("\nStep 3: Cleaning data...")
initial_count = len(df)

# Filtering out duplicates and invalid senders.
df = df.drop_duplicates(subset=["message_id"], keep="first")
df = df[(df["from"] != "") & (df["date"] != "")]
df["from"] = df["from"].str.lower()
df = df[df["from"].str.match(EMAIL_PATTERN).fillna(False)]

# Timestamp normalization.
clean_dates = df["date"].str.replace(r"\s*\([A-Z]{3,4}\)\s*$", "", regex=True)
df["timestamp"] = pd.to_datetime(clean_dates, errors="coerce", utc=True).dt.tz_localize(None)
df = df[(df["timestamp"] >= pd.Timestamp("1995-01-01")) &
        (df["timestamp"] <= pd.Timestamp("2005-12-31"))]

# Extracting basic temporal features (Year, Month, etc.)
for attr in ["year", "month", "day", "hour", "weekday"]:
    df[attr] = getattr(df["timestamp"].dt, attr)

# Body cleaning stages (Matching your layered approach)
body = df["body_raw"].astype(str)

# Stage 1: Remove original/forwarded markers
body = body.str.replace(r"-+\s*Original Message\s*-+.*$", "", regex=True, flags=re.DOTALL | re.IGNORECASE)
body = body.str.replace(r"-+\s*Forwarded by.*?-+", "", regex=True, flags=re.DOTALL | re.IGNORECASE)
df["body_stage1"] = body.str.strip()

# Stage 2: Remove quoted reply lines (lines starting with '>')
df["body_stage2"] = df["body_stage1"].apply(
    lambda text: "\n".join([line for line in text.split("\n") if not line.strip().startswith(">")]).strip()
)

# Stage 3: Normalize excessive newlines
df["body_cleaned"] = df["body_stage2"].apply(
    lambda x: re.sub(r"\n{3,}", "\n\n", str(x)).strip() if x else ""
)

print(f"Cleaned to {len(df):,} emails (removed {initial_count - len(df):,}).")
print("\nSample of cleaned data:")
df[["from", "timestamp", "body_cleaned"]].head()


Step 3: Cleaning data...
Cleaned to 46,420 emails (removed 457,147).

Sample of cleaned data:


,from,timestamp,body_cleaned
0,phillip.allen@enron.com,2001-05-14 23:39:00,Here is our forecast
17,phillip.allen@enron.com,2001-05-04 18:26:00,"Tim,\n\nmike grigsby is having problems with a..."
28,phillip.allen@enron.com,2001-05-03 22:57:00,"Reagan,\n\nJust wanted to give you an update. ..."
39,phillip.allen@enron.com,2001-05-02 19:36:00,"Jim,\n\nIs there going to be a conference call..."
50,phillip.allen@enron.com,2001-05-02 17:27:00,Ina Rangel\n05/01/2001 12:24 PM\nTo:\tPhillip ...


In [7]:
# Enrichment Features - include in step 3
print("\nAdding analytical enrichment features...")

# Calculating text metrics from the cleaned body.
df["email_length"] = df["body_cleaned"].str.len().fillna(0)
df["word_count"] = df["body_cleaned"].str.split().str.len().fillna(0)

# Apply Word Count Threshold
df = df[(df["word_count"] >= 50) & (df["word_count"] <= 500)]

# Topic Classification Setup
CATEGORIES = {
    "Legal & Regulatory": ["subpoena", "law", "litigation", "regulatory", "ferc", "sec", "compliance", "court", "legal", "attorney", "counsel", "agreement", "contract"],
    "Financial & Accounting": ["mtm", "spe", "accounting", "debt", "asset", "finance", "hedge", "audit", "balance", "capital", "expenditure", "revenue", "loss", "profit"],
    "Energy Trading": ["desk", "wholesale", "grid", "star", "boy", "shorty", "power", "transmission", "gas", "electricity", "market", "trading", "commodity", "price"],
    "HR & Internal": ["feedback", "performance", "hr", "training", "announcement", "insurance", "payroll", "retirement", "benefits", "vacation", "holiday", "meeting", "interview", "staff", "management"],
    "News & Media": ["newsletter", "news", "press", "release", "daily", "update", "media", "report"]
}


# Classified each email by CATEGORIES
def classify_text(text):
    if not isinstance(text, str):
        return "Other"
    text = text.lower()
    for category, keywords in CATEGORIES.items():
        for keyword in keywords:
            if re.search(r'\b' + re.escape(keyword) + r'\b', text):
                return category
    return "Other"

# Assign Topics
df["category"] = df["body_cleaned"].apply(classify_text)

# Removing 'Other' category
df = df[df["category"] != "Other"]

# Categorizing communication by time of day.
time_map = {range(5, 12): "Morning", range(12, 17): "Afternoon", range(17, 21): "Evening"}
df["communication_time_category"] = df["hour"].apply(
    lambda h: next((v for k, v in time_map.items() if h in k), "Night")
)


# Text Tokenization (Removing stopwords and punctuation)
df["body_tokenized"] = df["body_cleaned"].apply(
    lambda text: " ".join([t for t in re.findall(r"\b[a-zA-Z]+\b", str(text).lower()) 
                          if t not in STOPWORDS and len(t) > 2]) if text else ""
)

print("Sample enrichment features:")
df[["email_length", "word_count", "communication_time_category", "body_tokenized","category"]].head()


Adding analytical enrichment features...
Sample enrichment features:


,email_length,word_count,communication_time_category,body_tokenized,category
28,465,82,Night,reagan wanted give update changed unit mix inc...,News & Media
50,1357,212,Evening,ina rangel phillip allen hou ect ect subject s...,HR & Internal
178,332,62,Afternoon,mary add balances together total spread months...,HR & Internal
184,2083,245,Morning,capacitycenter com marketing capacitycenter co...,Legal & Regulatory
186,831,128,Morning,jeffrey shankman phillip allen hou ect ect kei...,Energy Trading


In [8]:
# --- Step 4: Relationship Extraction ---
print("\nStep 4: Extracting recipient relationships (To, Cc, Bcc)...")
rel_data = df[["message_id", "timestamp", "from", "to", "x_cc", "x_bcc"]].copy()

# Combine all recipient fields into a single column for explosion
rel_data["all_recipients_raw"] = (
    rel_data["to"].fillna("") + "," + 
    rel_data["x_cc"].fillna("") + "," + 
    rel_data["x_bcc"].fillna("")
)

# Exploding comma and semicolon separated lists of recipients into individual rows.
rel_data["recipient_list"] = rel_data["all_recipients_raw"].str.split(r"[,;]")
exploded_df = rel_data.explode("recipient_list")

exploded_df["recipient"] = exploded_df["recipient_list"].str.strip().str.lower()
exploded_df = exploded_df[exploded_df["recipient"].str.match(EMAIL_PATTERN).fillna(False)]

comms_df = exploded_df[["from", "recipient", "message_id", "timestamp"]].rename(
    columns={"from": "sender_email", "recipient": "receiver_email"}
).drop_duplicates()

# Classify as 'internal' if both sender and receiver are within Enron, 'external' otherwise.
comms_df["communication_type"] = comms_df.apply(
    lambda row: "internal" if row["sender_email"].endswith("@enron.com") and 
                            row["receiver_email"].endswith("@enron.com") else "external",
    axis=1
)


# --- Aggregated Communications --- include in step 4
print("\nCreating aggregated communication summaries...")
# Building a summary table for every sender-receiver pair.
agg_comms = comms_df.groupby(["sender_email", "receiver_email"]).agg({
    "timestamp": ["min", "max", "count"],
    "communication_type": "first"
}).reset_index()

# Flattening the multi-index columns.
agg_comms.columns = ["sender_email", "receiver_email", "first_contact", "last_contact", 
                     "communication_frequency", "communication_type"]
agg_comms["temporal_span_days"] = (agg_comms["last_contact"] - agg_comms["first_contact"]).dt.days

print("Sample aggregated communications:")
agg_comms.head()


Step 4: Extracting recipient relationships (To, Cc, Bcc)...

Creating aggregated communication summaries...
Sample aggregated communications:


,sender_email,receiver_email,first_contact,last_contact,communication_frequency,communication_type,temporal_span_days
0,1.10043390.-13@multexinvestornetwork.com,jwillia@enron.com,2001-05-09 14:52:46,2001-05-09 14:52:46,1,external,0
1,11-jribnick@interchange-energy.com,tdonoho@enron.com,2001-05-10 16:08:05,2001-05-10 16:08:05,1,external,0
2,12-jribnick@interchange-energy.com,tdonoho@enron.com,2001-05-17 14:37:08,2001-05-17 14:37:08,1,external,0
3,1800flowers.86665396@s2u2.com,lcampbel@enron.com,2001-05-04 17:09:00,2001-05-04 17:09:00,3,external,0
4,1800flowers.88555715@s2u2.com,lcampbel@enron.com,2001-05-08 17:13:00,2001-05-08 17:13:00,2,external,0


In [9]:
# --- Step 5: Entity Creation ---
print("\nStep 5: Generating participant table...")
unique_emails = pd.concat([comms_df["sender_email"], comms_df["receiver_email"]]).unique()
employees_df = pd.DataFrame({"email_address": unique_emails})
employees_df["employee_id"] = employees_df["email_address"].apply(lambda x: hashlib.md5(x.encode()).hexdigest()[:16])

# Mapping parsed names from X-From headers back to email addresses.
discovered_names = df[df["x_from"] != ""].groupby("from")["x_from"].first().reset_index()
employees_df = employees_df.merge(discovered_names, left_on="email_address", right_on="from", how="left")
employees_df["name"] = employees_df["x_from"].fillna("Unknown")

employees_df = employees_df[["employee_id", "email_address", "name"]].sort_values("email_address")
print(f"Formed {len(employees_df):,} employee/entity records.")
print("\nSample participant list:")
employees_df.head()


Step 5: Generating participant table...
Formed 4,214 employee/entity records.

Sample participant list:


,employee_id,email_address,name
1813,5c4cfede0bb88903,1.10043390.-13@multexinvestornetwork.com,Multex Investor <1.10043390.-13@multexinvestor...
324,6fcef0744ba520c4,11-jribnick@interchange-energy.com,"""Jerry A. Ribnick"" <11-jribnick@interchange-en..."
327,4b96ec00ebe1e113,12-jribnick@interchange-energy.com,Jerry Ribnick <12-jribnick@interchange-energy....
133,b8466a738bdae75e,1800flowers.86665396@s2u2.com,"""1-800-FLOWERS.COM"" <1800Flowers.86665396@s2u2..."
136,348c8743e8b7a43a,1800flowers.88555715@s2u2.com,"""1-800-FLOWERS.COM"" <1800Flowers.88555715@s2u2..."


In [10]:
# --- Employee Metrics --- include in step 5
print("\nComputing per-employee communication metrics...")
# Measuring the activity of each individual in the network based on cleaned data.

# 1. Message counts
sent_counts = df.groupby("from").size().reset_index(name="emails_sent_count")
recv_counts = comms_df.groupby("receiver_email").size().reset_index(name="emails_received_count")

# 2. Unique contact counts (sent to and received from)
unique_sent_to = comms_df.groupby("sender_email")["receiver_email"].nunique().reset_index(name="sent_to_count")
unique_recv_from = comms_df.groupby("receiver_email")["sender_email"].nunique().reset_index(name="received_from_count")

# 3. Internal vs External contact reach
internal_comms = comms_df[comms_df["communication_type"] == "internal"]
external_comms = comms_df[comms_df["communication_type"] == "external"]

int_contacts = internal_comms.groupby("sender_email")["receiver_email"].nunique().reset_index(name="internal_contacts_sent")
ext_contacts = external_comms.groupby("sender_email")["receiver_email"].nunique().reset_index(name="external_contacts_sent")


# Merge all metrics into a single table
metrics_df = employees_df.copy()
merge_targets = [
    (sent_counts, "from"), 
    (recv_counts, "receiver_email"), 
    (unique_sent_to, "sender_email"), 
    (unique_recv_from, "receiver_email"),
    (int_contacts, "sender_email"),
    (ext_contacts, "sender_email")
]

for target_df, target_col in merge_targets:
    metrics_df = metrics_df.merge(target_df, left_on="email_address", right_on=target_col, how="left").drop(columns=[target_col])

# Cleaning up metric columns: fill NaNs with 0 and ensure integer types.
metric_cols = ["emails_sent_count", "emails_received_count", "sent_to_count", 
               "received_from_count", "internal_contacts_sent", "external_contacts_sent"]
metrics_df[metric_cols] = metrics_df[metric_cols].fillna(0).astype(int)

# Total unique contacts combined
metrics_df["unique_contacts_total"] = metrics_df["sent_to_count"] + metrics_df["received_from_count"]

print("Sample employee metrics (top 5 by sent count):")
metrics_df.sort_values("emails_sent_count", ascending=False)
metrics_df.head()


Computing per-employee communication metrics...
Sample employee metrics (top 5 by sent count):


,employee_id,email_address,name,emails_sent_count,emails_received_count,sent_to_count,received_from_count,internal_contacts_sent,external_contacts_sent,unique_contacts_total
0,5c4cfede0bb88903,1.10043390.-13@multexinvestornetwork.com,Multex Investor <1.10043390.-13@multexinvestor...,1,0,1,0,0,1,1
1,6fcef0744ba520c4,11-jribnick@interchange-energy.com,"""Jerry A. Ribnick"" <11-jribnick@interchange-en...",1,0,1,0,0,1,1
2,4b96ec00ebe1e113,12-jribnick@interchange-energy.com,Jerry Ribnick <12-jribnick@interchange-energy....,1,0,1,0,0,1,1
3,b8466a738bdae75e,1800flowers.86665396@s2u2.com,"""1-800-FLOWERS.COM"" <1800Flowers.86665396@s2u2...",3,0,1,0,0,1,1
4,348c8743e8b7a43a,1800flowers.88555715@s2u2.com,"""1-800-FLOWERS.COM"" <1800Flowers.88555715@s2u2...",2,0,1,0,0,1,1


In [11]:
# --- Comprehensive Validation --- Just checking
print("\nRunning final data validation...")
validation_passed = True
validation_errors = []

# 1. Structural Checks
required_stages = ['body_raw', 'body_stage1', 'body_stage2', 'body_cleaned']
if not all(col in df.columns for col in required_stages):
    validation_errors.append("Layered cleaning stages missing from email table.")

# 2. Data Integrity Checks
if df["message_id"].duplicated().sum() > 0:
    validation_errors.append(f"Duplicate Message-IDs detected ({df['message_id'].duplicated().sum():,}).")

if df["timestamp"].isna().sum() > 0:
    validation_errors.append("Found emails with missing or invalid timestamps.")

# 3. Domain/Logical Checks
if df['year'].min() < 1995 or df['year'].max() > 2005:
    validation_errors.append(f"Date range mismatch: found emails from {df['year'].min()} to {df['year'].max()}.")

if not employees_df['email_address'].str.match(EMAIL_PATTERN).all():
    validation_errors.append("Invalid email formats detected in employee table.")

# 4. Metrics & Export Checks
required_metrics = ['emails_sent_count', 'emails_received_count', 'unique_contacts_total']
if not all(col in metrics_df.columns for col in required_metrics):
    validation_errors.append("Core employee metrics are missing.")


# Summary Output
if not validation_errors:
    print("All core integrity and structural checks passed.")
else:
    print("Validation found issues:")
    for err in validation_errors:
        print(f"  - {err}")
    validation_passed = False

# Print high-level metrics (Matching your original pipeline log style)
print(f"\nFinal Metrics Summary:")
print(f" - Total Valid Emails: {len(df):,}")
print(f" - Total Unique Participants: {len(employees_df):,}")
print(f" - Total Directional Relationships: {len(comms_df):,}")
if 'communication_type' in comms_df.columns:
    internal = (comms_df['communication_type'] == 'internal').sum()
    print(f" - Internal (Enron-to-Enron): {internal:,}")
    print(f" - External: {len(comms_df) - internal:,}")




Running final data validation...
All core integrity and structural checks passed.

Final Metrics Summary:
 - Total Valid Emails: 19,271
 - Total Unique Participants: 4,214
 - Total Directional Relationships: 30,161
 - Internal (Enron-to-Enron): 21,379
 - External: 8,782


In [12]:
employees_df.head()

,employee_id,email_address,name
1813,5c4cfede0bb88903,1.10043390.-13@multexinvestornetwork.com,Multex Investor <1.10043390.-13@multexinvestor...
324,6fcef0744ba520c4,11-jribnick@interchange-energy.com,"""Jerry A. Ribnick"" <11-jribnick@interchange-en..."
327,4b96ec00ebe1e113,12-jribnick@interchange-energy.com,Jerry Ribnick <12-jribnick@interchange-energy....
133,b8466a738bdae75e,1800flowers.86665396@s2u2.com,"""1-800-FLOWERS.COM"" <1800Flowers.86665396@s2u2..."
136,348c8743e8b7a43a,1800flowers.88555715@s2u2.com,"""1-800-FLOWERS.COM"" <1800Flowers.88555715@s2u2..."


In [13]:
# cleaned Email saved into email.csv
df[["message_id", "timestamp", "from", "to", "subject", "body_cleaned", "year", "month", "day", "hour", "weekday", "category"]].head()

,message_id,timestamp,from,to,subject,body_cleaned,year,month,day,hour,weekday,category
28,<8572706.1075855378498.JavaMail.evans@thyme>,2001-05-03 22:57:00,phillip.allen@enron.com,rlehmann@yahoo.com,Mime-Version: 1.0,"Reagan,\n\nJust wanted to give you an update. ...",2001,5,3,22,3,News & Media
50,<27936946.1075855378542.JavaMail.evans@thyme>,2001-05-02 17:27:00,phillip.allen@enron.com,tori.kuykendall@enron.com,Re: 2- SURVEY - PHILLIP ALLEN,Ina Rangel\n05/01/2001 12:24 PM\nTo:\tPhillip ...,2001,5,2,17,2,HR & Internal
178,<18351800.1075855690826.JavaMail.evans@thyme>,2000-05-26 14:19:00,phillip.allen@enron.com,maryrichards7@hotmail.com,Re: balance on truck/loan,"Mary,\n\nIf we add both balances together the ...",2000,5,26,14,4,HR & Internal
184,<6057369.1075855690933.JavaMail.evans@thyme>,2000-05-22 11:44:00,phillip.allen@enron.com,"matthew.lenhart@enron.com, jane.tholt@enron.co...",Gas Transportation Market Intelligence,"""CapacityCenter.com"" <marketing@capacitycenter...",2000,5,22,11,0,Legal & Regulatory
186,<22039237.1075855690977.JavaMail.evans@thyme>,2000-05-19 10:46:00,phillip.allen@enron.com,tim.belden@enron.com,Large Deal Alert,From: Jeffrey A Shankman ...,2000,5,19,10,4,Energy Trading


In [14]:
# Communications
comms_df.head()

,sender_email,receiver_email,message_id,timestamp,communication_type
28,phillip.allen@enron.com,rlehmann@yahoo.com,<8572706.1075855378498.JavaMail.evans@thyme>,2001-05-03 22:57:00,external
50,phillip.allen@enron.com,tori.kuykendall@enron.com,<27936946.1075855378542.JavaMail.evans@thyme>,2001-05-02 17:27:00,internal
178,phillip.allen@enron.com,maryrichards7@hotmail.com,<18351800.1075855690826.JavaMail.evans@thyme>,2000-05-26 14:19:00,external
184,phillip.allen@enron.com,matthew.lenhart@enron.com,<6057369.1075855690933.JavaMail.evans@thyme>,2000-05-22 11:44:00,internal
184,phillip.allen@enron.com,jane.tholt@enron.com,<6057369.1075855690933.JavaMail.evans@thyme>,2000-05-22 11:44:00,internal


In [15]:
# Aggregated Communnication
agg_comms.head()

,sender_email,receiver_email,first_contact,last_contact,communication_frequency,communication_type,temporal_span_days
0,1.10043390.-13@multexinvestornetwork.com,jwillia@enron.com,2001-05-09 14:52:46,2001-05-09 14:52:46,1,external,0
1,11-jribnick@interchange-energy.com,tdonoho@enron.com,2001-05-10 16:08:05,2001-05-10 16:08:05,1,external,0
2,12-jribnick@interchange-energy.com,tdonoho@enron.com,2001-05-17 14:37:08,2001-05-17 14:37:08,1,external,0
3,1800flowers.86665396@s2u2.com,lcampbel@enron.com,2001-05-04 17:09:00,2001-05-04 17:09:00,3,external,0
4,1800flowers.88555715@s2u2.com,lcampbel@enron.com,2001-05-08 17:13:00,2001-05-08 17:13:00,2,external,0


In [16]:
# Email enrichment features saved into email_enrichment_features.csv
df[["message_id", "email_length", "word_count", "communication_time_category", "body_tokenized", "category"]].head()

,message_id,email_length,word_count,communication_time_category,body_tokenized,category
28,<8572706.1075855378498.JavaMail.evans@thyme>,465,82,Night,reagan wanted give update changed unit mix inc...,News & Media
50,<27936946.1075855378542.JavaMail.evans@thyme>,1357,212,Evening,ina rangel phillip allen hou ect ect subject s...,HR & Internal
178,<18351800.1075855690826.JavaMail.evans@thyme>,332,62,Afternoon,mary add balances together total spread months...,HR & Internal
184,<6057369.1075855690933.JavaMail.evans@thyme>,2083,245,Morning,capacitycenter com marketing capacitycenter co...,Legal & Regulatory
186,<22039237.1075855690977.JavaMail.evans@thyme>,831,128,Morning,jeffrey shankman phillip allen hou ect ect kei...,Energy Trading


In [17]:
# Employee Metrics
metrics_df.head()

,employee_id,email_address,name,emails_sent_count,emails_received_count,sent_to_count,received_from_count,internal_contacts_sent,external_contacts_sent,unique_contacts_total
0,5c4cfede0bb88903,1.10043390.-13@multexinvestornetwork.com,Multex Investor <1.10043390.-13@multexinvestor...,1,0,1,0,0,1,1
1,6fcef0744ba520c4,11-jribnick@interchange-energy.com,"""Jerry A. Ribnick"" <11-jribnick@interchange-en...",1,0,1,0,0,1,1
2,4b96ec00ebe1e113,12-jribnick@interchange-energy.com,Jerry Ribnick <12-jribnick@interchange-energy....,1,0,1,0,0,1,1
3,b8466a738bdae75e,1800flowers.86665396@s2u2.com,"""1-800-FLOWERS.COM"" <1800Flowers.86665396@s2u2...",3,0,1,0,0,1,1
4,348c8743e8b7a43a,1800flowers.88555715@s2u2.com,"""1-800-FLOWERS.COM"" <1800Flowers.88555715@s2u2...",2,0,1,0,0,1,1


In [18]:
# --- Step 6: Final Data Export ---
print(f"\nStep 6: Exporting all 6 datasets to '{OUTPUT_DIR}'...")
os.makedirs(OUTPUT_DIR, exist_ok=True)


Step 6: Exporting all 6 datasets to 'final_datasets'...


In [19]:
export_mappings = {
    "employees.csv": employees_df,
    "emails_cleaned.csv": df[["message_id", "timestamp", "from", "to", "subject", "body_cleaned", "year", "month", "day", "hour", "weekday", "category"]],
    "communications.csv": comms_df,
    "aggregated_communications.csv": agg_comms,
    "email_enrichment_features.csv": df[["message_id", "email_length", "word_count", "communication_time_category", "body_tokenized", "category"]],
    "employee_metrics.csv": metrics_df
}

In [20]:
for filename, data in export_mappings.items():
    data.to_csv(os.path.join(OUTPUT_DIR, filename), index=False)
    print(f"Saved {filename}")

print("\n--- Pipeline Complete! All datasets are ready. ---")

Saved employees.csv
Saved emails_cleaned.csv
Saved communications.csv
Saved aggregated_communications.csv
Saved email_enrichment_features.csv
Saved employee_metrics.csv

--- Pipeline Complete! All datasets are ready. ---


In [21]:
# Save a markdown version of the topic summary for easy viewing
summary = df['category'].value_counts().reset_index()
summary.columns = ['Category', 'Number of Emails']
with open(os.path.join(LOG_DIR, "topic_summary.md"), "w") as f:
    f.write("# Email Topic Distribution Summary\n\n")
    f.write("| Category | Number of Emails |\n")
    f.write("| :--- | :--- |\n")
    for _, row in summary.iterrows():
        f.write(f"| {row['Category']} | {row['Number of Emails']} |\n")
print(f"Saved topic_summary.md to {LOG_DIR}")

Saved topic_summary.md to logs


In [22]:
# --- Generate Dataset Inventory Summary ---
print("Generating dataset_info.md...")
inventory_path = os.path.join(LOG_DIR, "dataset_info.md")
with open(inventory_path, "w") as f:
    f.write("# Final Datasets Inventory\n\n")
    f.write("This log provides an overview of all datasets generated by the pipeline.\n\n")
    
    for filename, data in export_mappings.items():
        f.write(f"### {filename}\n")
        f.write(f"- **Total Records:** {len(data):,}\n")
        f.write(f"- **Columns:** `{', '.join(data.columns)}`\n\n")

print(f"Saved dataset_info.md to {LOG_DIR}")

Generating dataset_info.md...
Saved dataset_info.md to logs


Create Sample Datasets

In [23]:
def create_sample_dataset(source_dir="final_datasets", output_dir="sample_email_by_category", sample_size_per_category=20):
    """
    Creates a smaller, consistent sample dataset from the full datasets.
    It samples a fixed number of emails per category and filters all related files
    (employees, communications, metrics, etc.) to match the sampled emails.
    """
    print(f"Starting sample dataset creation. Sampling {sample_size_per_category} emails per category...")
    os.makedirs(output_dir, exist_ok=True)
    
    # 1. Load and sample emails
    emails_path = os.path.join(source_dir, "emails_cleaned.csv")
    if not os.path.exists(emails_path):
        print(f"Error: {emails_path} not found.")
        return
        
    df_emails = pd.read_csv(emails_path)
    
    # Sample linearly by category
    # If a category has fewer than 'sample_size_per_category', it takes all of them.
    df_sampled_emails = df_emails.groupby("category", group_keys=False).apply(
        lambda x: x.sample(min(len(x), sample_size_per_category), random_state=42)
    )
    
    sampled_message_ids = set(df_sampled_emails["message_id"].unique())
    
    # Collect all unique emails (senders + all recipients) from the sampled set
    active_email_addresses = set(df_sampled_emails["from"].dropna().unique())
    # Recipients can be multiple separated by comma
    for to_field in df_sampled_emails["to"].dropna():
        for email in str(to_field).split(","):
            email = email.strip()
            if email:
                active_email_addresses.add(email)
                
    print(f"Sampled {len(df_sampled_emails)} total emails.")
    print(f"Identified {len(active_email_addresses)} unique participants.")

    # 2. Save sampled emails
    df_sampled_emails.to_csv(os.path.join(output_dir, "sample_email.csv"), index=False)
    
    # 3. Filter and save Enrichment Features
    enrichment_path = os.path.join(source_dir, "email_enrichment_features.csv")
    if os.path.exists(enrichment_path):
        df_enrich = pd.read_csv(enrichment_path)
        df_sampled_enrich = df_enrich[df_enrich["message_id"].isin(sampled_message_ids)]
        df_sampled_enrich.to_csv(os.path.join(output_dir, "sample_email_enrichment_features.csv"), index=False)
        print(f"Filtered email_enrichment_features.csv -> {len(df_sampled_enrich)} rows.")
        
    # 4. Filter and save Communications
    comms_path = os.path.join(source_dir, "communications.csv")
    if os.path.exists(comms_path):
        df_comms = pd.read_csv(comms_path)
        df_sampled_comms = df_comms[df_comms["message_id"].isin(sampled_message_ids)]
        df_sampled_comms.to_csv(os.path.join(output_dir, "sample_communications.csv"), index=False)
        print(f"Filtered communications.csv -> {len(df_sampled_comms)} rows.")
        
    # 5. Filter and save Employees
    emp_path = os.path.join(source_dir, "employees.csv")
    if os.path.exists(emp_path):
        df_emp = pd.read_csv(emp_path)
        df_sampled_emp = df_emp[df_emp["email_address"].isin(active_email_addresses)]
        df_sampled_emp.to_csv(os.path.join(output_dir, "sample_employees.csv"), index=False)
        print(f"Filtered employees.csv -> {len(df_sampled_emp)} rows.")

    # 6. Filter and save Employee Metrics
    metrics_path = os.path.join(source_dir, "employee_metrics.csv")
    if os.path.exists(metrics_path):
        df_metrics = pd.read_csv(metrics_path)
        df_sampled_metrics = df_metrics[df_metrics["email_address"].isin(active_email_addresses)]
        df_sampled_metrics.to_csv(os.path.join(output_dir, "sample_employee_metrics.csv"), index=False)
        print(f"Filtered employee_metrics.csv -> {len(df_sampled_metrics)} rows.")
        
    # 7. Filter and save Aggregated Communications
    agg_comms_path = os.path.join(source_dir, "aggregated_communications.csv")
    if os.path.exists(agg_comms_path):
        df_agg = pd.read_csv(agg_comms_path)
        # Keep edges where both sender and receiver are in our active list
        df_sampled_agg = df_agg[
            df_agg["sender_email"].isin(active_email_addresses) & 
            df_agg["receiver_email"].isin(active_email_addresses)
        ]
        df_sampled_agg.to_csv(os.path.join(output_dir, "sample_aggregated_communications.csv"), index=False)
        print(f"Filtered aggregated_communications.csv -> {len(df_sampled_agg)} rows.")

    print(f"\nSuccess! All sampled datasets saved to '{output_dir}/'")

In [24]:
if __name__ == "__main__":
    # Give the sample size of each category
    create_sample_dataset(sample_size_per_category=200)

Starting sample dataset creation. Sampling 200 emails per category...
Sampled 1000 total emails.
Identified 989 unique participants.
Filtered email_enrichment_features.csv -> 1000 rows.
Filtered communications.csv -> 1451 rows.
Filtered employees.csv -> 974 rows.
Filtered employee_metrics.csv -> 974 rows.
Filtered aggregated_communications.csv -> 2345 rows.

Success! All sampled datasets saved to 'sample_email_by_category/'
